# 뛰어야산다2 방송 효과 심층 분석
> 모든 데이터 소스 통합 + 교란변수 완전 통제

**방송 정보:**
- 프로그램: MBN 뛰어야산다2 (스포츠리얼버라이어티)
- 촬영: 2025-11-16~17 (1박2일)
- 방영: 2026-01-12 (월) 22:10
- 출연: 션, 이영표, 양세형, 배성재, 이기광
- 노출: 신정호정원, 곡교천 은행나무길, 현충사, 온천, 캠핑장
- 타겟: 2030세대 / 시청률: 1.5%
- 예산: 4,500만원

**데이터 레이어:**
1. T맵 내비게이션 → 관광지 방문 (2019~2025)
2. SKT 유동인구 → 실제 인구 유입 (2026-01~02)
3. 아산페이 → 소비/매출 (2026-01~02)
4. 네이버 검색 트렌드 → 온라인 관심도
5. 날씨 → 교란변수 통제
6. 공휴일/연휴/요일 → 교란변수 통제

**분석 방법:**
- DID (이중차분): 방송노출 지역 vs 미노출
- CausalImpact: "방송이 없었다면" 합성 시나리오
- 다중회귀: 날씨+요일+연휴 통제 후 순수 방송 더미 효과

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib import font_manager, rc
import zipfile, os, re, io, glob, json, time
from pathlib import Path
from datetime import datetime, timedelta
import httpx
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

try:
    rc('font', family=font_manager.FontProperties(fname='C:/Windows/Fonts/malgun.ttf').get_name())
except:
    plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

COLORS = ['#2196F3','#FF5722','#4CAF50','#FFC107','#9C27B0','#00BCD4','#E91E63']
print('Ready')

In [ ]:
# ============================
# 설정
# ============================
DATA_DIR = Path(r'C:\Users\HP\Desktop\01.데이터')
TMAP_DIR = DATA_DIR / '04. 내비게이션 데이터'
POP_DIR  = DATA_DIR / '01. 인구 데이터'
OUTPUT_DIR = Path(r'C:\Users\HP\Desktop\02.분석결과')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# .env
for p in [Path('..') / 'crawlers' / '.env', Path('.') / '.env']:
    if p.exists(): load_dotenv(p); break

# 방송 정보
AIR_DATE = pd.Timestamp('2026-01-12')
BROADCAST = '뛰어야산다2'
EXPOSED_SPOTS = ['신정호', '곡교천', '현충사']  # T맵 매칭용
EXPOSED_DONGS = ['온양5동', '염치읍', '온양1동']  # 처치군
CONTROL_DONGS = ['배방읍', '탕정면', '둔포면', '신창면', '송악면']  # 대조군 (방송 미노출)

# 분석 윈도우
PRE_START  = pd.Timestamp('2025-12-13')  # 방송 전 30일
PRE_END    = pd.Timestamp('2026-01-11')  # 방송 전날
POST_START = pd.Timestamp('2026-01-13')  # 방송 다음날
POST_END   = pd.Timestamp('2026-02-11')  # 방송 후 30일

print(f'방송: {BROADCAST} | 방영일: {AIR_DATE.date()}')
print(f'분석: {PRE_START.date()} ~ {POST_END.date()}')
print(f'처치군: {EXPOSED_DONGS}')
print(f'대조군: {CONTROL_DONGS}')

---
## 1. 교란변수 수집: 날씨 + 공휴일

In [ ]:
# 1-1. 기상청 ASOS 관측 데이터 (아산 인근: 천안 관측소 232)
# 공공데이터포털 or 기상자료개방포털에서 다운 가능
# 여기서는 Open-Meteo API 사용 (무료, API키 불필요)

def get_weather(lat, lon, start_date, end_date):
    """Open-Meteo API로 일별 날씨 수집"""
    url = 'https://archive-api.open-meteo.com/v1/archive'
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start_date, 'end_date': end_date,
        'daily': 'temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,rain_sum,snowfall_sum,wind_speed_10m_max',
        'timezone': 'Asia/Seoul',
    }
    resp = httpx.get(url, params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    df = pd.DataFrame(data['daily'])
    df['time'] = pd.to_datetime(df['time'])
    return df

# 아산시 좌표 (온양온천역 기준)
ASAN_LAT, ASAN_LON = 36.7806, 127.0042

print('날씨 수집 중...')
df_weather = get_weather(ASAN_LAT, ASAN_LON, 
                          PRE_START.strftime('%Y-%m-%d'), 
                          POST_END.strftime('%Y-%m-%d'))
df_weather.columns = ['date','temp_max','temp_min','temp_mean','precip','rain','snow','wind_max']

# 관광 불리 날씨 플래그
df_weather['bad_weather'] = ((df_weather['rain'] > 5) | (df_weather['snow'] > 1) | 
                              (df_weather['temp_min'] < -10)).astype(int)
df_weather['freezing'] = (df_weather['temp_min'] < -5).astype(int)

print(f'날씨: {len(df_weather)}일, {df_weather["date"].min().date()} ~ {df_weather["date"].max().date()}')
print(f'악천후 일수: {df_weather["bad_weather"].sum()}일')
print(f'한파(영하5도 이하): {df_weather["freezing"].sum()}일')
display(df_weather.describe().round(1))

In [ ]:
# 1-2. 공휴일/연휴/요일
dates = pd.date_range(PRE_START, POST_END, freq='D')
df_cal = pd.DataFrame({'date': dates})
df_cal['dow'] = df_cal['date'].dt.dayofweek  # 0=월 6=일
df_cal['is_weekend'] = (df_cal['dow'] >= 5).astype(int)
df_cal['dow_name'] = df_cal['date'].dt.day_name()

# 2026년 1~2월 공휴일
holidays_2026 = [
    '2026-01-01',  # 신정
    '2026-02-16', '2026-02-17', '2026-02-18',  # 설연휴
    '2026-03-01',  # 삼일절 (범위 밖이지만)
]
df_cal['is_holiday'] = df_cal['date'].isin(pd.to_datetime(holidays_2026)).astype(int)
df_cal['is_holiday_or_weekend'] = ((df_cal['is_weekend'] == 1) | (df_cal['is_holiday'] == 1)).astype(int)

# 설연휴 구간 (전후 포함)
df_cal['is_seol'] = ((df_cal['date'] >= '2026-02-14') & (df_cal['date'] <= '2026-02-19')).astype(int)

# 방송 전후 구분
df_cal['period'] = 'pre'
df_cal.loc[df_cal['date'] == AIR_DATE, 'period'] = 'air_date'
df_cal.loc[df_cal['date'] > AIR_DATE, 'period'] = 'post'
df_cal['post'] = (df_cal['date'] > AIR_DATE).astype(int)

# 날씨 조인
df_cal = df_cal.merge(df_weather, on='date', how='left')

print(f'달력: {len(df_cal)}일')
print(f'  방송 전: {(df_cal["period"]=="pre").sum()}일 | 방송 후: {(df_cal["period"]=="post").sum()}일')
print(f'  주말/공휴일: {df_cal["is_holiday_or_weekend"].sum()}일')
print(f'  설연휴: {df_cal["is_seol"].sum()}일')
display(df_cal.head())

---
## 2. 레이어1: T맵 관광지 방문

In [ ]:
# T맵 로드 + 관광지 매칭
POI_KEYWORDS = {
    '신정호': ['신정호', '신정호정원'], '현충사': ['현충사'],
    '곡교천': ['곡교천', '은행나무길'], '온양온천': ['온양온천', '온천', '족욕'],
    '도고온천': ['도고', '도고파라다이스'], '외암민속마을': ['외암', '외암민속'],
    '피나클랜드': ['피나클랜드'],
}
def match_poi(name):
    if pd.isna(name): return None
    s = str(name).lower()
    for poi, kws in POI_KEYWORDS.items():
        for kw in kws:
            if kw in s: return poi
    return None

tmap_files = sorted(glob.glob(str(TMAP_DIR / '**' / 'as_tmap_od_*.csv'), recursive=True))
print(f'T맵 파일: {len(tmap_files)}개')

chunks = []
for f in tqdm(tmap_files, desc='T맵'):
    for enc in ['utf-8-sig','cp949']:
        try: chunks.append(pd.read_csv(f, encoding=enc)); break
        except: continue

df_tmap = pd.concat(chunks, ignore_index=True)
df_tmap['drv_ymd'] = pd.to_datetime(df_tmap['drv_ymd'].astype(str))
df_tmap['matched_poi'] = df_tmap['dstn_nm'].apply(match_poi)
df_tmap['is_tourism'] = df_tmap['dstn_ctgy'].str.startswith('여행/레저').fillna(False)
print(f'T맵: {len(df_tmap):,}행 | {df_tmap["drv_ymd"].min().date()} ~ {df_tmap["drv_ymd"].max().date()}')

In [ ]:
# 뛰어야산다2 노출 관광지 일별 시계열
exposed_mask = df_tmap['matched_poi'].isin(EXPOSED_SPOTS)
non_exposed_mask = df_tmap['matched_poi'].notna() & ~exposed_mask

# 처치군: 방송에 나온 관광지
treat_tmap = df_tmap[exposed_mask].groupby('drv_ymd').agg(
    visits=('vst_cnt','sum'), avg_stay=('avg_stay_min','mean')
).reset_index().rename(columns={'drv_ymd': 'date'})
treat_tmap['group'] = 'exposed'

# 대조군: 방송에 안 나온 관광지
ctrl_tmap = df_tmap[non_exposed_mask].groupby('drv_ymd').agg(
    visits=('vst_cnt','sum'), avg_stay=('avg_stay_min','mean')
).reset_index().rename(columns={'drv_ymd': 'date'})
ctrl_tmap['group'] = 'control'

tmap_daily = pd.concat([treat_tmap, ctrl_tmap], ignore_index=True)

# 전년 동기 비교용 (2024-12-13 ~ 2025-02-11 vs 2025-12-13 ~ 2026-02-11)
yoy_start = PRE_START - pd.Timedelta(days=365)
yoy_end = POST_END - pd.Timedelta(days=365)

print(f'처치군(노출관광지): {len(treat_tmap):,}일')
print(f'대조군(미노출관광지): {len(ctrl_tmap):,}일')
print(f'전년동기: {yoy_start.date()} ~ {yoy_end.date()}')

In [ ]:
# T맵 시각화: 처치군 vs 대조군 + 전년 동기
fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=False)

for ax, group, title in [
    (axes[0], 'exposed', f'방송 노출 관광지 ({"+".join(EXPOSED_SPOTS)})'),
    (axes[1], 'control', '미노출 관광지 (대조군)'),
]:
    gdf = tmap_daily[tmap_daily['group'] == group]
    
    # 올해
    current = gdf[(gdf['date'] >= PRE_START - timedelta(days=30)) & 
                   (gdf['date'] <= POST_END + timedelta(days=30))]
    ax.plot(current['date'], current['visits'], alpha=0.3, color='#2196F3')
    if len(current) > 7:
        ax.plot(current['date'], current['visits'].rolling(7,min_periods=1).mean(),
                color='#2196F3', linewidth=2, label='2025-26 (7일MA)')
    
    # 전년
    prev = gdf[(gdf['date'] >= yoy_start - timedelta(days=30)) & 
               (gdf['date'] <= yoy_end + timedelta(days=30))].copy()
    if len(prev) > 0:
        prev['date_shifted'] = prev['date'] + pd.Timedelta(days=365)
        ax.plot(prev['date_shifted'], prev['visits'].rolling(7,min_periods=1).mean(),
                color='#9E9E9E', linewidth=1.5, linestyle='--', label='전년동기 (7일MA)')
    
    ax.axvline(x=AIR_DATE, color='red', linewidth=2, linestyle='--', label=f'방영일 ({AIR_DATE.date()})')
    ax.axvspan(PRE_START, PRE_END, alpha=0.05, color='blue', label='방송 전')
    ax.axvspan(POST_START, POST_END, alpha=0.05, color='red', label='방송 후')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle(f'{BROADCAST} 방송 전후 T맵 관광지 방문 비교', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / f'fig_deep_{BROADCAST}_tmap.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. 레이어2: SKT 유동인구

In [ ]:
# SKT 유동인구 로드 (2026-01, 02)
# 핵심 파일: AS_SKT_SX_AGE_DONG → 읍면동별 성별/연령별 일별 유동인구

skt_files = sorted(glob.glob(str(POP_DIR / '**' / 'AS_SKT_SX_AGE_DONG_*.csv'), recursive=True))
print(f'SKT 읍면동 파일: {len(skt_files)}개')
for f in skt_files: print(f'  {os.path.basename(f)} ({os.path.getsize(f)/1024/1024:.0f}MB)')

if skt_files:
    skt_chunks = []
    for f in tqdm(skt_files, desc='SKT 읍면동'):
        for enc in ['utf-8-sig','cp949']:
            try: skt_chunks.append(pd.read_csv(f, encoding=enc)); break
            except: continue
    df_skt_dong = pd.concat(skt_chunks, ignore_index=True)
    print(f'SKT 읍면동: {len(df_skt_dong):,}행')
    print(f'컬럼: {list(df_skt_dong.columns)}')
    display(df_skt_dong.head(3))
else:
    df_skt_dong = pd.DataFrame()
    print('SKT 읍면동 파일 없음')

In [ ]:
# SKT 유입(Inflow) 데이터: 어디서 아산으로 왔는지
skt_inflow = sorted(glob.glob(str(POP_DIR / '**' / 'AS_SKT_AGE_UNQ_OUTFLOW_NOPE_*.csv'), recursive=True))
# 또는 SGG 단위 유입
skt_sgg = sorted(glob.glob(str(POP_DIR / '**' / 'AS_SKT_SGG_UNQ_VST_NOPE_*.csv'), recursive=True))

print(f'SKT 유출입: {len(skt_inflow)}개')
print(f'SKT 시군구 방문: {len(skt_sgg)}개')

if skt_sgg:
    sgg_chunks = []
    for f in skt_sgg:
        for enc in ['utf-8-sig','cp949']:
            try: sgg_chunks.append(pd.read_csv(f, encoding=enc)); break
            except: continue
    df_skt_sgg = pd.concat(sgg_chunks, ignore_index=True)
    print(f'\nSKT 시군구 방문: {len(df_skt_sgg):,}행')
    print(f'컬럼: {list(df_skt_sgg.columns)}')
    display(df_skt_sgg.head(3))
else:
    df_skt_sgg = pd.DataFrame()

In [ ]:
# SKT 데이터에서 처치군/대조군 일별 유동인구 추출
# (컬럼명은 실제 데이터에 맞게 조정 필요)

if len(df_skt_dong) > 0:
    # 날짜 컬럼 확인
    date_col = [c for c in df_skt_dong.columns if 'YMD' in c.upper() or 'DATE' in c.upper()]
    dong_col = [c for c in df_skt_dong.columns if 'DONG' in c.upper()]
    pop_cols = [c for c in df_skt_dong.columns if 'PPLTN' in c.upper() or 'CNT' in c.upper()]
    
    print(f'날짜 컬럼: {date_col}')
    print(f'동 컬럼: {dong_col}')
    print(f'인구 컬럼: {pop_cols[:5]}...')
    
    # 아산시 읍면동 코드 → 이름 매핑
    DONG_MAP = {
        '4420025000': '염치읍', '4420025300': '배방읍', '4420031000': '송악면',
        '4420033000': '탕정면', '4420034000': '신창면', '4420035000': '음봉면',
        '4420036000': '둔포면', '4420037000': '영인면', '4420038000': '인주면',
        '4420039000': '선장면', '4420040000': '도고면',
        '4420057000': '온양1동', '4420058000': '온양2동', '4420059000': '온양3동',
        '4420060000': '온양4동', '4420061000': '온양5동', '4420062000': '온양6동',
    }
    
    if dong_col:
        df_skt_dong['dong_name'] = df_skt_dong[dong_col[0]].astype(str).map(DONG_MAP)
        df_skt_dong['is_exposed'] = df_skt_dong['dong_name'].isin(EXPOSED_DONGS)
        df_skt_dong['is_control'] = df_skt_dong['dong_name'].isin(CONTROL_DONGS)
        
        print(f'\n매핑된 행: {df_skt_dong["dong_name"].notna().sum():,}')
        print(f'처치군: {df_skt_dong["is_exposed"].sum():,}')
        print(f'대조군: {df_skt_dong["is_control"].sum():,}')

---
## 4. 레이어3: 아산페이 매출

In [ ]:
# 아산페이 ZIP 로드
def extract_asanpay_zip(zip_path):
    dfs = {}
    with zipfile.ZipFile(zip_path) as zf:
        for info in zf.infolist():
            try: name = info.filename.encode('cp437').decode('cp949')
            except: name = info.filename
            if not name.endswith('.xlsx'): continue
            data = zf.read(info.filename)
            xl = pd.ExcelFile(io.BytesIO(data))
            for sheet in xl.sheet_names:
                key = f"{name}|{sheet}"
                dfs[key] = pd.read_excel(io.BytesIO(data), sheet_name=sheet)
                print(f"  {name} > {sheet}: {dfs[key].shape}")
    return dfs

asanpay_zips = sorted(DATA_DIR.glob('아산페이*.zip'))
print(f'아산페이 ZIP: {len(asanpay_zips)}개')

all_pay = {}
for zp in asanpay_zips:
    print(f'\n=== {zp.name} ===')
    month = '01' if '1월' in zp.name else '02' if '2' in zp.name else 'unknown'
    for k, v in extract_asanpay_zip(zp).items():
        all_pay[f"{month}|{k}"] = v

In [ ]:
# 가맹점 마스터 + 결제 데이터 조인
merchant_list, payment_list, purchase_list = [], [], []

for key, df in all_pay.items():
    month = key.split('|')[0]
    cols = ' '.join(str(c) for c in df.columns)
    df_copy = df.copy()
    df_copy['_month'] = month
    
    if '가맹점ID' in cols and '총금액' in cols:
        payment_list.append(df_copy)
    elif '가맹점ID' in cols and '행정동' in cols:
        merchant_list.append(df_copy)
    elif '연령대' in cols and '구매금액' in cols:
        purchase_list.append(df_copy)

# 가맹점 마스터
if merchant_list:
    df_merchant = pd.concat(merchant_list).drop_duplicates(subset='가맹점ID', keep='last')
    df_merchant['dong'] = df_merchant.apply(
        lambda r: r['행정동'] if pd.notna(r['행정동']) and r['행정동'] != '[NULL]'
        else (re.search(r'아산시\s+(\S+[읍면동])', str(r.get('기본주소',''))).group(1) 
              if re.search(r'아산시\s+(\S+[읍면동])', str(r.get('기본주소',''))) else None),
        axis=1
    )
    print(f'가맹점: {len(df_merchant):,}개')

# 결제 데이터
if payment_list and merchant_list:
    df_payment = pd.concat(payment_list, ignore_index=True)
    lookup = df_merchant[['가맹점ID','dong','분류']].drop_duplicates(subset='가맹점ID')
    df_pay = df_payment.merge(lookup, on='가맹점ID', how='left')
    for c in ['총금액','총결제건수','QR결제금액','카드결제금액']:
        if c in df_pay.columns:
            df_pay[c] = pd.to_numeric(df_pay[c], errors='coerce').fillna(0)
    df_pay['is_exposed'] = df_pay['dong'].isin(EXPOSED_DONGS)
    df_pay['is_control'] = df_pay['dong'].isin(CONTROL_DONGS)
    print(f'결제: {len(df_pay):,}행 | 읍면동 매핑: {df_pay["dong"].notna().mean():.1%}')

# 연령대별 구매
if purchase_list:
    df_purchase = pd.concat(purchase_list, ignore_index=True)
    print(f'연령별 구매: {len(df_purchase):,}행')

---
## 5. 레이어4: 네이버 검색 트렌드

In [ ]:
# 네이버 데이터랩 - 뛰어야산다 관련 키워드
NAVER_ID = os.getenv('NAVER_CLIENT_ID', '')
NAVER_SECRET = os.getenv('NAVER_CLIENT_SECRET', '')

def naver_trend(keywords, start, end):
    resp = httpx.post('https://openapi.naver.com/v1/datalab/search',
        json={'startDate': start, 'endDate': end, 'timeUnit': 'date',
              'keywordGroups': [{'groupName': k, 'keywords': [k]} for k in keywords[:5]]},
        headers={'X-Naver-Client-Id': NAVER_ID, 'X-Naver-Client-Secret': NAVER_SECRET,
                 'Content-Type': 'application/json'}, timeout=30)
    rows = []
    for r in resp.json().get('results', []):
        for pt in r.get('data', []):
            rows.append({'keyword': r['title'], 'date': pt['period'], 'ratio': pt['ratio']})
    return pd.DataFrame(rows)

if NAVER_ID:
    kws = ['뛰어야산다 아산', '아산 여행', '신정호정원', '현충사 아산', '곡교천']
    df_trend = naver_trend(kws, PRE_START.strftime('%Y-%m-%d'), POST_END.strftime('%Y-%m-%d'))
    df_trend['date'] = pd.to_datetime(df_trend['date'])
    print(f'검색 트렌드: {len(df_trend):,}행')
else:
    # 이미 수집된 파일에서 로드
    try:
        df_trend = pd.read_csv(OUTPUT_DIR / 'naver_search_trends.csv', encoding='utf-8-sig')
        df_trend['date'] = pd.to_datetime(df_trend['date'])
        df_trend = df_trend[df_trend['broadcast'] == '뛰어야산다2']
        print(f'기존 트렌드 로드: {len(df_trend):,}행')
    except:
        df_trend = pd.DataFrame()
        print('검색 트렌드 없음')

---
## 6. 통합 분석: 교란변수 통제 + 순수 효과 추정

In [ ]:
# 6-1. T맵 DID: 날씨+요일 통제
# 노출 관광지 vs 미노출 관광지의 방송 전후 차이

tmap_panel = tmap_daily.merge(df_cal[['date','post','is_weekend','is_holiday_or_weekend',
                                       'is_seol','temp_mean','bad_weather','freezing','dow']],
                               on='date', how='inner')
tmap_panel['treat'] = (tmap_panel['group'] == 'exposed').astype(int)
tmap_panel['did'] = tmap_panel['post'] * tmap_panel['treat']

# 분석 윈도우만
tmap_panel = tmap_panel[(tmap_panel['date'] >= PRE_START) & (tmap_panel['date'] <= POST_END)]

print(f'T맵 DID 패널: {len(tmap_panel):,}행')
print(f'\n그룹별 평균 방문:')
display(tmap_panel.groupby(['group', 'post'])['visits'].mean().unstack().round(1))

# 단순 DID
means = tmap_panel.groupby(['post', 'treat'])['visits'].mean()
try:
    did_simple = (means[(1,1)] - means[(0,1)]) - (means[(1,0)] - means[(0,0)])
    print(f'\n단순 DID 효과: {did_simple:+,.1f}건/일')
except:
    print('DID 계산 불가 (데이터 부족)')

In [ ]:
# 6-2. 다중회귀 DID: 날씨+요일+공휴일 통제
try:
    from statsmodels.formula.api import ols
    
    # 모델1: 단순 DID
    m1 = ols('visits ~ post * treat', data=tmap_panel).fit()
    
    # 모델2: 날씨 + 요일 통제
    m2 = ols('visits ~ post * treat + temp_mean + bad_weather + is_weekend + is_seol', 
             data=tmap_panel).fit()
    
    # 모델3: 요일 더미 전부
    m3 = ols('visits ~ post * treat + temp_mean + bad_weather + C(dow) + is_seol + freezing', 
             data=tmap_panel).fit()
    
    print('=== T맵 DID 회귀분석 결과 ===')
    print(f'\n{"모델":10s} | {"DID효과(post:treat)":>20s} | {"p-value":>10s} | {"R2":>8s}')
    print('-' * 60)
    for name, model in [('단순 DID', m1), ('+날씨+주말', m2), ('+요일더미', m3)]:
        coef = model.params.get('post:treat', np.nan)
        pval = model.pvalues.get('post:treat', np.nan)
        print(f'{name:10s} | {coef:>20,.1f} | {pval:>10.4f} | {model.rsquared:>8.3f}')
    
    print(f'\n--- 최종 모델(모델3) 상세 ---')
    print(m3.summary().tables[1])
    
except ImportError:
    print('statsmodels 필요: pip install statsmodels')

In [ ]:
# 6-3. 아산페이 DID
if 'df_pay' in dir() and len(df_pay) > 0:
    # 월별 그룹 비교
    pay_group = df_pay.groupby(['_month', 'is_exposed']).agg(
        total_amt=('총금액', 'sum'),
        total_cnt=('총결제건수', 'sum'),
        avg_amt=('총금액', 'mean'),
        merchants=('가맹점ID', 'nunique'),
    ).reset_index()
    
    print('아산페이 그룹별 비교:')
    display(pay_group)
    
    # DID 계산 (가맹점당 평균 매출 기준)
    try:
        t_01 = pay_group[(pay_group['_month']=='01') & (pay_group['is_exposed']==True)]['avg_amt'].values[0]
        t_02 = pay_group[(pay_group['_month']=='02') & (pay_group['is_exposed']==True)]['avg_amt'].values[0]
        c_01 = pay_group[(pay_group['_month']=='01') & (pay_group['is_exposed']==False)]['avg_amt'].values[0]
        c_02 = pay_group[(pay_group['_month']=='02') & (pay_group['is_exposed']==False)]['avg_amt'].values[0]
        
        did_pay = (t_02 - t_01) - (c_02 - c_01)
        print(f'\n아산페이 DID 효과: {did_pay:+,.0f}원/가맹점')
        print(f'  처치군: {t_01:,.0f} → {t_02:,.0f} ({t_02-t_01:+,.0f})')
        print(f'  대조군: {c_01:,.0f} → {c_02:,.0f} ({c_02-c_01:+,.0f})')
    except:
        print('DID 계산 실패')
else:
    print('아산페이 데이터 없음')

In [ ]:
# 6-4. 연령대별 소비 변화 (타겟 매칭: 뛰어야산다 = 2030)
if 'df_purchase' in dir() and len(df_purchase) > 0:
    age_col = [c for c in df_purchase.columns if '연령' in str(c)][0]
    amt_col = [c for c in df_purchase.columns if '구매금액' in str(c)][0]
    
    age_pivot = df_purchase.pivot_table(values=amt_col, index=age_col, columns='_month', 
                                         aggfunc='sum', fill_value=0)
    if '01' in age_pivot.columns and '02' in age_pivot.columns:
        age_pivot['변화율(%)'] = ((age_pivot['02'] - age_pivot['01']) / age_pivot['01'] * 100).round(1)
    
    print('연령대별 구매금액 변화:')
    display(age_pivot)
    
    # 시각화
    fig, ax = plt.subplots(figsize=(10, 5))
    age_pivot[['01','02']].plot(kind='bar', ax=ax, color=['#2196F3','#FF5722'])
    ax.set_title(f'{BROADCAST} 방송 후 연령대별 소비 변화', fontweight='bold')
    ax.set_ylabel('구매금액')
    ax.legend(['1월 (방송 전반)', '2월 (방송 후)'])
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
    ax.grid(True, alpha=0.3, axis='y')
    # 2030 하이라이트
    for i, label in enumerate(age_pivot.index):
        if '20' in str(label) or '30' in str(label):
            ax.get_children()[i].set_edgecolor('red')
            ax.get_children()[i].set_linewidth(2)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'fig_deep_{BROADCAST}_age.png', dpi=150, bbox_inches='tight')
    plt.show()

---
## 7. 종합 대시보드

In [ ]:
# 종합 4패널 차트
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# (1) 검색 트렌드
ax = axes[0][0]
if len(df_trend) > 0:
    for kw in df_trend['keyword'].unique():
        kdf = df_trend[df_trend['keyword']==kw].sort_values('date')
        ax.plot(kdf['date'], kdf['ratio'], label=kw, linewidth=1.5)
    ax.axvline(x=AIR_DATE, color='red', linestyle='--', linewidth=2)
ax.set_title('1. 검색 트렌드 (네이버)', fontweight='bold')
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

# (2) T맵 관광지 방문
ax = axes[0][1]
for grp, color, label in [('exposed','#FF5722','노출 관광지'), ('control','#2196F3','미노출 관광지')]:
    gdf = tmap_daily[(tmap_daily['group']==grp) & 
                      (tmap_daily['date']>=PRE_START) & (tmap_daily['date']<=POST_END)]
    if len(gdf) > 7:
        ax.plot(gdf['date'], gdf['visits'].rolling(7,min_periods=1).mean(), 
                color=color, linewidth=2, label=label)
ax.axvline(x=AIR_DATE, color='red', linestyle='--', linewidth=2)
ax.set_title('2. T맵 관광지 방문 (7일MA)', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# (3) 날씨 (교란변수)
ax = axes[1][0]
ax.fill_between(df_cal['date'], df_cal['temp_min'], df_cal['temp_max'], alpha=0.3, color='#2196F3')
ax.plot(df_cal['date'], df_cal['temp_mean'], color='#2196F3', linewidth=1.5, label='평균기온')
rain_days = df_cal[df_cal['bad_weather']==1]
ax.scatter(rain_days['date'], rain_days['temp_mean'], color='red', s=30, zorder=5, label='악천후')
ax.axvline(x=AIR_DATE, color='red', linestyle='--', linewidth=2)
ax.axhline(y=0, color='black', linewidth=0.5, linestyle=':')
ax.set_title('3. 날씨 (교란변수)', fontweight='bold')
ax.set_ylabel('기온 (C)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# (4) 아산페이 매출
ax = axes[1][1]
if 'df_pay' in dir() and len(df_pay) > 0:
    grp = df_pay.groupby(['_month', 'is_exposed'])['총금액'].sum().reset_index()
    grp['label'] = grp['is_exposed'].map({True: '노출지역', False: '기타'})
    pv = grp.pivot(index='label', columns='_month', values='총금액').fillna(0)
    pv.plot(kind='bar', ax=ax, color=['#2196F3','#FF5722'])
    ax.legend(['1월', '2월'])
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.set_title('4. 아산페이 매출', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

plt.suptitle(f'{BROADCAST} 방송효과 종합 분석 (방영: {AIR_DATE.date()})',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / f'fig_deep_{BROADCAST}_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 최종 요약 리포트
print('=' * 65)
print(f'  {BROADCAST} 방송 홍보효과 심층 분석 결과')
print('=' * 65)
print(f'  방영일: {AIR_DATE.date()} | 시청률: 1.5% | 예산: 4,500만원')
print(f'  노출 관광지: {", ".join(EXPOSED_SPOTS)}')
print(f'  분석 기간: {PRE_START.date()} ~ {POST_END.date()}')
print('-' * 65)

print(f'\n  [1] 온라인 관심도 (네이버 검색)')
if len(df_trend) > 0:
    for kw in df_trend['keyword'].unique():
        kdf = df_trend[df_trend['keyword']==kw]
        pre_avg = kdf[kdf['date'] < AIR_DATE]['ratio'].mean()
        post_avg = kdf[kdf['date'] >= AIR_DATE]['ratio'].mean()
        lift = ((post_avg-pre_avg)/pre_avg*100) if pre_avg > 0 else float('inf')
        print(f'    {kw}: {pre_avg:.1f} → {post_avg:.1f} ({lift:+.1f}%)')

print(f'\n  [2] 관광지 방문 (T맵)')
for grp in ['exposed', 'control']:
    gdf = tmap_panel[tmap_panel['group']==grp]
    pre_avg = gdf[gdf['post']==0]['visits'].mean()
    post_avg = gdf[gdf['post']==1]['visits'].mean()
    label = '노출관광지' if grp=='exposed' else '대조군'
    print(f'    {label}: {pre_avg:.1f} → {post_avg:.1f}건/일')

print(f'\n  [3] DID 효과 (교란변수 통제 후)')
try:
    coef = m3.params.get('post:treat', np.nan)
    pval = m3.pvalues.get('post:treat', np.nan)
    sig = 'O' if pval < 0.05 else 'X'
    print(f'    T맵 DID: {coef:+,.1f}건/일 (p={pval:.4f}, 유의: {sig})')
    print(f'    통제변수: 기온, 악천후, 요일, 설연휴, 한파')
except:
    pass

print(f'\n  [4] 아산페이 DID')
try:
    print(f'    순수 효과: {did_pay:+,.0f}원/가맹점')
except:
    print(f'    (데이터 부족)')

print(f'\n  [5] 교란변수 통제 현황')
print(f'    날씨: Open-Meteo API (기온/강수/적설/풍속)')
print(f'    공휴일: 신정(1/1), 설연휴(2/16~18)')
print(f'    요일: 월~일 더미변수')
print(f'    계절: 1월 비수기 → 교란 최소')
print(f'    동시 방송: 없음 (단독)')

print(f'\n  [6] 한계점')
print(f'    - 아산페이는 월별 집계 → 일별 변화 추적 불가')
print(f'    - T맵 데이터가 2025-12까지 → 방송 후(2026-01) 데이터 없을 수 있음')
print(f'    - 1월 방영이라 비수기 효과(기저 낮음) 고려 필요')
print('=' * 65)

In [ ]:
# 결과 저장
print('생성된 파일:')
for pattern in [f'*deep_{BROADCAST}*', 'asanpay*']:
    for f in sorted(OUTPUT_DIR.glob(pattern)):
        try: print(f'  {f.name} ({f.stat().st_size/1024:.0f}KB)')
        except: pass